In [4]:
#!/usr/bin/env python3
"""
Visualize the ESP32 Experiment 12 synthetic point cloud generator.

Direct Python port of generateFrame() from src/main.cpp. Uses the same
xorshift32 PRNG, the same per-frame seed derivation, the same stable
per-identity body offsets (mix32 hash), the same sensor biases, the same
trajectories, and the same RNG-consumption order.

Fidelity notes
--------------
* The xorshift32 state is integer-only.  The sequence of random draws is
  byte-identical between the ESP32 and this script.
* C uses `float` (32-bit IEEE 754); Python uses `double` (64-bit).
  Individual coordinates may differ by ~1e-7 m.  Point clouds are
  visually identical and the cluster structure is identical.
* The sensor 3-D coordinates are metadata only (see README §4.2).  They
  are stored here so you can plot them, but they do NOT affect point
  generation.

Usage
-----
    python visualize_points.py                               # WALK, seed 0, frame 0
    python visualize_points.py --scenario CROSSING           # crossing, frame 0
    python visualize_points.py --frame 75                    # WALK frame 75 (t = 7.5 s)
    python visualize_points.py --scenario MERGE --frame 150  # merge, t = 15 s
    python visualize_points.py --drop 0.3 --frame 40         # 30 % group dropout
    python visualize_points.py --grid 9                      # 3x3 grid of frames
    python visualize_points.py --scenario CROSSING --grid 12 --save cross.png
"""

import argparse
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401  (registers 3-D projection)

# ============================================================================
# Config constants — identical to src/main.cpp
# ============================================================================
SENSOR_COUNT         = 3
HUMAN_COUNT          = 2
PTS_PER_HUMAN_SENSOR = 40
NOISE_PTS_PER_SENSOR = 8

DT          = 0.1
BODY_EXT_XY = 0.12
BODY_EXT_Z  = 0.20

DEFAULT_SEED          = 0
DEFAULT_SIGMA         = 0.05
DEFAULT_DROP_P        = 0.0
DEFAULT_FRAME         = 0
DEFAULT_SCENARIO      = "WALK"
DEFAULT_FRAMES_PER_RUN = 300   # matches src/main.cpp

# ============================================================================
# PRNG — xorshift32, matches ESP32 code exactly
# ============================================================================
class Rng:
    def __init__(self):
        self.state = 0xA53C91E7

    def _next(self):
        x = self.state
        x = (x ^ ((x << 13) & 0xFFFFFFFF)) & 0xFFFFFFFF
        x = (x ^ (x >> 17)) & 0xFFFFFFFF
        x = (x ^ ((x << 5)  & 0xFFFFFFFF)) & 0xFFFFFFFF
        self.state = x
        return x

    def uniform01(self):
        return (self._next() & 0x00FFFFFF) / 16777216.0

    def uniform_range(self, lo, hi):
        return lo + (hi - lo) * self.uniform01()

    def gaussian_noise(self):
        # Sum of 12 uniforms minus 6 — exactly as on the ESP32.
        s = 0.0
        for _ in range(12):
            s += self.uniform01()
        return s - 6.0

    def seed_for_frame(self, seed, frame_index):
        # Matches:
        #   rngState = 0xA53C91E7u
        #            ^ (uint32_t)(seed * 1597334677u)
        #            ^ (uint32_t)(frameIndex * 2654435761u);
        a = 0xA53C91E7
        b = (seed * 1597334677) & 0xFFFFFFFF
        c = (frame_index * 2654435761) & 0xFFFFFFFF
        self.state = (a ^ b ^ c) & 0xFFFFFFFF


# ============================================================================
# mix32 / stableOffset — exact integer reproduction
# ============================================================================
def mix32(x):
    x &= 0xFFFFFFFF
    x ^= x >> 16
    x = (x * 0x7FEB352D) & 0xFFFFFFFF
    x ^= x >> 15
    x = (x * 0x846CA68B) & 0xFFFFFFFF
    x ^= x >> 16
    return x & 0xFFFFFFFF


def stable_offset(identity, salt, extent):
    h = mix32((identity * 2654435761 + salt) & 0xFFFFFFFF)
    u = (h & 0x00FFFFFF) / 16777216.0
    return (u * 2.0 - 1.0) * extent


# ============================================================================
# Sensors — matches initSensors() in src/main.cpp
#   (position is metadata; bias and sigma are used by generateFrame)
# ============================================================================
SENSORS = [
    #  pos (x, y, z)         bias
    ((-4.0,  0.0, 3.0),  0.005),   # s = 0
    (( 0.0,  4.0, 3.5), -0.008),   # s = 1
    (( 3.5, -3.0, 3.0),  0.004),   # s = 2
]


# ============================================================================
# Trajectories — direct ports of trajWalk / trajCrossing / trajMerge
# ============================================================================
def traj_walk(human, t):
    speed = 0.10
    x = -1.0 + speed * t
    y = -1.5 if human == 0 else 1.5
    y += 0.03 * math.sin(0.5 * t + human * 1.7)
    z = 0.90 if human == 0 else 0.95
    return (x, y, z)


def traj_crossing(human, t):
    T = max(0.0, t)
    phase = T / 15.0 if T < 15.0 else 1.0
    if human == 0:
        x = -1.5 + 3.0 * phase
        y = 0.05 * math.sin(0.4 * T)
        z = 0.90
    else:
        y = -1.5 + 3.0 * phase
        x = -0.05 * math.sin(0.4 * T)
        z = 0.95
    return (x, y, z)


def traj_merge(human, t):
    T = t
    ySign = -1.0 if human == 0 else 1.0
    if T < 10.0:
        p = T / 10.0
        x = (-2.0 + 2.0 * p) if human == 0 else (2.0 - 2.0 * p)
        y = ySign * (1.2 - 1.1 * p)
    elif T < 20.0:
        x = 0.0
        y = ySign * 0.10
    else:
        p = (T - 20.0) / 10.0
        x = (-2.0 * p) if human == 0 else (2.0 * p)
        y = ySign * (0.10 + 1.1 * p)
    z = 0.90 if human == 0 else 0.95
    return (x, y, z)


TRAJECTORIES = {
    "WALK":     traj_walk,
    "CROSSING": traj_crossing,
    "MERGE":    traj_merge,
}


# ============================================================================
# generateFrame — identical layout and RNG-consumption order
# ============================================================================
def generate_frame(scenario, seed, frame_index,
                   sigma=DEFAULT_SIGMA, drop_p=DEFAULT_DROP_P):
    """
    Returns an (N, 5) float32 array:
        cols 0..2 : (x, y, z) in metres
        col 3     : sensor index 0..2
        col 4     : human index 0..1, or -1 for ambient noise
    """
    rng = Rng()
    rng.seed_for_frame(seed, frame_index)
    t = frame_index * DT
    traj = TRAJECTORIES[scenario]

    rows = []

    # --- body points, matching the ESP32 loop nest ---------------------
    for s in range(SENSOR_COUNT):
        s_bias = SENSORS[s][1]
        for h in range(HUMAN_COUNT):
            # Match C: only draw if drop_p > 0, and consume the draw
            # BEFORE deciding whether to skip this (s, h) block.
            if drop_p > 0.0:
                if rng.uniform01() < drop_p:
                    continue

            cx, cy, cz = traj(h, t)
            for k in range(PTS_PER_HUMAN_SENSOR):
                ident = (s * HUMAN_COUNT + h) * PTS_PER_HUMAN_SENSOR + k
                bx = cx + stable_offset(ident, 0x11, BODY_EXT_XY)
                by = cy + stable_offset(ident, 0x22, BODY_EXT_XY)
                bz = cz + stable_offset(ident, 0x33, BODY_EXT_Z)
                x = bx + s_bias + sigma * rng.gaussian_noise()
                y = by + s_bias + sigma * rng.gaussian_noise()
                z = bz + s_bias + sigma * rng.gaussian_noise()
                rows.append((x, y, z, s, h))

    # --- ambient noise points ------------------------------------------
    for s in range(SENSOR_COUNT):
        for _ in range(NOISE_PTS_PER_SENSOR):
            x = rng.uniform_range(-3.5, 3.5)
            y = rng.uniform_range(-3.5, 3.5)
            z = rng.uniform_range(0.2, 2.0)
            rows.append((x, y, z, s, -1))

    return np.array(rows, dtype=np.float32)


def ground_truth(scenario, frame_index):
    t = frame_index * DT
    traj = TRAJECTORIES[scenario]
    return [traj(h, t) for h in range(HUMAN_COUNT)]


# ============================================================================
# Plotting helpers
# ============================================================================
HUM_COLOR   = {0: "#d62728", 1: "#1f77b4"}    # red / blue
SENS_MARKER = {0: "o",       1: "s",       2: "^"}   # circle / square / triangle


def _scatter(ax, xs, ys, zs, color, marker, size, alpha, label=None, mode="3d"):
    """Dispatch to 2-D or 3-D scatter."""
    if mode == "3d":
        ax.scatter(xs, ys, zs, c=color, marker=marker, s=size,
                   alpha=alpha, edgecolors="none", label=label)
    elif mode == "xy":
        ax.scatter(xs, ys, c=color, marker=marker, s=size,
                   alpha=alpha, edgecolors="none", label=label)
    elif mode == "xz":
        ax.scatter(xs, zs, c=color, marker=marker, s=size,
                   alpha=alpha, edgecolors="none", label=label)
    elif mode == "yz":
        ax.scatter(ys, zs, c=color, marker=marker, s=size,
                   alpha=alpha, edgecolors="none", label=label)


def draw_cloud(ax, pts, ground, *, marker_scale=1.0, show_gt=True,
               label_groups=False, mode="3d"):
    """Draw a point cloud on ax (2-D or 3-D) with per-group colouring."""
    # Body points — grouped by (sensor, human).
    for s in range(SENSOR_COUNT):
        for h in range(HUMAN_COUNT):
            mask = (pts[:, 3] == s) & (pts[:, 4] == h)
            if not mask.any():
                continue
            p = pts[mask]
            lbl = f"S{s} H{h}" if label_groups else None
            _scatter(ax, p[:, 0], p[:, 1], p[:, 2],
                     color=HUM_COLOR[h], marker=SENS_MARKER[s],
                     size=16 * marker_scale, alpha=0.75,
                     label=lbl, mode=mode)

    # Ambient noise points — small grey dots.
    mask = pts[:, 4] == -1
    if mask.any():
        p = pts[mask]
        lbl = "noise" if label_groups else None
        _scatter(ax, p[:, 0], p[:, 1], p[:, 2],
                 color="#cccccc", marker=".", size=6, alpha=0.55,
                 label=lbl, mode=mode)

    # Ground-truth markers — large hollow X.
    if show_gt:
        for h, (gx, gy, gz) in enumerate(ground):
            lbl = f"GT H{h}" if label_groups else None
            if mode == "3d":
                ax.scatter([gx], [gy], [gz], c=HUM_COLOR[h], marker="X",
                           s=200, edgecolors="black", linewidths=1.6,
                           label=lbl)
            elif mode == "xy":
                ax.scatter([gx], [gy], c=HUM_COLOR[h], marker="X",
                           s=180, edgecolors="black", linewidths=1.4,
                           label=lbl)
            elif mode == "xz":
                ax.scatter([gx], [gz], c=HUM_COLOR[h], marker="X",
                           s=180, edgecolors="black", linewidths=1.4,
                           label=lbl)
            elif mode == "yz":
                ax.scatter([gy], [gz], c=HUM_COLOR[h], marker="X",
                           s=180, edgecolors="black", linewidths=1.4,
                           label=lbl)


# ============================================================================
# Figures
# ============================================================================
def show_single_frame(scenario, seed, frame_index, sigma, drop_p, save=None):
    pts = generate_frame(scenario, seed, frame_index, sigma, drop_p)
    ground = ground_truth(scenario, frame_index)

    fig = plt.figure(figsize=(15, 12))

    ax3d = fig.add_subplot(2, 2, 1, projection="3d")
    ax_xy = fig.add_subplot(2, 2, 2)
    ax_xz = fig.add_subplot(2, 2, 3)
    ax_yz = fig.add_subplot(2, 2, 4)

    draw_cloud(ax3d, pts, ground, mode="3d", label_groups=True)
    draw_cloud(ax_xy, pts, ground, mode="xy")
    draw_cloud(ax_xz, pts, ground, mode="xz")
    draw_cloud(ax_yz, pts, ground, mode="yz")

    ax3d.set_xlabel("X (m)"); ax3d.set_ylabel("Y (m)"); ax3d.set_zlabel("Z (m)")
    ax3d.set_xlim(-4, 4); ax3d.set_ylim(-4, 4); ax3d.set_zlim(0.0, 2.5)
    ax3d.set_title(
        f"{scenario}  seed={seed}  frame={frame_index}  "
        f"sigma={sigma:.2f}  drop={drop_p:.2f}")
    ax3d.legend(loc="upper left", fontsize=7, ncol=2)

    for ax, xlab, ylab in ((ax_xy, "X", "Y"),
                           (ax_xz, "X", "Z"),
                           (ax_yz, "Y", "Z")):
        ax.set_xlabel(f"{xlab} (m)")
        ax.set_ylabel(f"{ylab} (m)")
        ax.set_xlim(-4, 4)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.3)

    ax_xy.set_title(f"XY projection (top view)   N = {len(pts)}")
    ax_xz.set_title("XZ projection (front view)")
    ax_yz.set_title("YZ projection (side view)")

    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
        print(f"saved -> {save}")
    plt.show()


def show_grid(scenario, seed, n_frames, sigma, drop_p, save=None):
    """Show n_frames evenly-spaced XY projections as a grid."""
    frame_indices = np.linspace(0, DEFAULT_FRAMES_PER_RUN - 1,
                                n_frames).astype(int)

    cols = int(math.ceil(math.sqrt(n_frames)))
    rows = int(math.ceil(n_frames / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.4 * rows),
                             squeeze=False)

    for i, fi in enumerate(frame_indices):
        r = i // cols
        c = i % cols
        ax = axes[r][c]

        pts = generate_frame(scenario, seed, int(fi), sigma, drop_p)
        ground = ground_truth(scenario, int(fi))

        draw_cloud(ax, pts, ground, mode="xy", marker_scale=0.75)

        ax.set_title(f"frame {fi:3d}  (t = {fi * DT:4.1f} s)", fontsize=9)
        ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)

    # Hide any unused subplot panels.
    for i in range(n_frames, rows * cols):
        r = i // cols
        c = i % cols
        axes[r][c].axis("off")

    fig.suptitle(f"{scenario}   seed = {seed}   sigma = {sigma}   "
                 f"dropout = {drop_p}", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.97))

    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
        print(f"saved -> {save}")
    plt.show()


# ============================================================================
# CLI
# ============================================================================
def main():
    ap = argparse.ArgumentParser(
        description="Visualize the ESP32 Experiment 12 synthetic point cloud "
                    "generator. Direct Python port of src/main.cpp.")
    ap.add_argument("--scenario", choices=list(TRAJECTORIES.keys()),
                    default=DEFAULT_SCENARIO,
                    help="Trajectory family (default: WALK)")
    ap.add_argument("--seed", type=int, default=DEFAULT_SEED,
                    help="RNG seed (default: 0)")
    ap.add_argument("--frame", type=int, default=DEFAULT_FRAME,
                    help="Frame index 0..299 (default: 0)")
    ap.add_argument("--sigma", type=float, default=DEFAULT_SIGMA,
                    help="Per-axis measurement std deviation in metres "
                         "(default: 0.05)")
    ap.add_argument("--drop", type=float, default=DEFAULT_DROP_P,
                    help="Per-sensor per-human group dropout probability "
                         "0..1 (default: 0.0)")
    ap.add_argument("--grid", type=int, default=0,
                    help="If > 0, plot N evenly-spaced frames as a grid "
                         "instead of one frame.")
    ap.add_argument("--save", default=None,
                    help="Save the figure to this path instead of showing it.")
    args = ap.parse_args()

    if args.grid > 0:
        show_grid(args.scenario, args.seed, args.grid,
                  args.sigma, args.drop, save=args.save)
    else:
        show_single_frame(args.scenario, args.seed, args.frame,
                          args.sigma, args.drop, save=args.save)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--scenario {WALK,CROSSING,MERGE}]
                             [--seed SEED] [--frame FRAME] [--sigma SIGMA]
                             [--drop DROP] [--grid GRID] [--save SAVE]
ipykernel_launcher.py: error: argument --frame: invalid int value: 'c:\\Users\\iCFixer.ir\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3706a52b8a4eb4d2e44dce9f91fe1549b77087db8.json'


SystemExit: 2

In [2]:
from IPython.display import HTML, display
from pathlib import Path

# ============ 1) لود فونت برای نمایش داخل Notebook ============
display(HTML("""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@400;600;800&display=swap" rel="stylesheet">
"""))

# ============ 2) محتوای HTML گزارش ============
html_content = """
<div style="
    direction: rtl;
    text-align: right;
    font-family: 'Vazirmatn', 'Sahel', 'IRANSans', 'Segoe UI', Tahoma, sans-serif;
    font-size: 17px;
    line-height: 2;
    color: #1f2937;
    background: #f9fafb;
    padding: 32px 40px;
    border-radius: 14px;
    border: 1px solid #e5e7eb;
    max-width: 950px;
    margin: 20px auto;
    box-shadow: 0 4px 14px rgba(0,0,0,0.06);
">

  <h1 style="font-size: 28px; font-weight: 800; color: #111827; border-bottom: 3px solid #3b82f6; padding-bottom: 12px; margin-bottom: 24px;">
    بنچمارک و بهینه‌سازی DBSCAN روی ESP32-S3
  </h1>

  <p style="font-size: 17px; color: #374151;">
    این پروژه عملکرد الگوریتم <b>DBSCAN سه‌بعدی</b> را روی <b>ESP32-S3 N16R8</b> بررسی می‌کند
    و از یک پیاده‌سازی پایه تا یک pipeline شبیه کاربرد واقعیِ سه‌سنسوره پیش می‌رود.
  </p>

  <h2 style="font-size: 22px; color: #1d4ed8; margin-top: 28px;">مشخصات کلی</h2>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>MCU: ESP32-S3</li>
    <li>فرکانس CPU: 240 MHz</li>
    <li>CPU: دو هسته</li>
    <li>Flash: 16 MB</li>
    <li>PSRAM: 8 MB</li>
    <li>Framework: Arduino</li>
    <li>Build System: PlatformIO</li>
    <li>داده: سه‌بعدی</li>
  </ul>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱. آزمایش Baseline Scaling</h1>
  <h3 style="font-size: 18px; color: #374151;">روند کار</h3>
  <p>
    در این مرحله DBSCAN به‌صورت brute-force اجرا شد. برای پیدا کردن همسایه‌های هر نقطه،
    کل dataset بررسی می‌شود. تعداد نقاط به‌صورت نمایی افزایش داده شد تا رفتار زمانی و حافظه مشخص شود.
    تعداد نقاط از 90 تا 23040 بررسی شد.
  </p>
  <h3 style="font-size: 18px; color: #374151;">نتیجه نهایی</h3>
  <p>پیاده‌سازی پایه تقریباً دارای این پیچیدگی است:</p>
  <pre style="direction: ltr; text-align: left; background: #111827; color: #f9fafb; padding: 14px 18px; border-radius: 8px; font-family: 'Fira Code', 'Consolas', monospace; font-size: 14.5px; line-height: 1.7; overflow-x: auto;">Time   ≈ O(N²)
Memory ≈ O(N)</pre>

  <p>در <code style="background:#eef2ff; padding:2px 6px; border-radius:4px;">N = 11520</code>:</p>
  <pre style="direction: ltr; text-align: left; background: #111827; color: #f9fafb; padding: 14px 18px; border-radius: 8px; font-family: 'Fira Code', 'Consolas', monospace; font-size: 14.5px; line-height: 1.7;">زمان اجرا ≈ 79.438 ثانیه
حافظه اصلی DBSCAN ≈ 225 KB
PSRAM آزاد ≈ 7.96 MB</pre>

  <p>
    بنابراین محدودیت نسخه پایه بیشتر <b>محاسباتی</b> است، نه کمبود حافظه.
    برای <code style="background:#eef2ff; padding:2px 6px; border-radius:4px;">N = 23040</code> اجرای کامل نهایی گزارش نشد.
  </p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۲. آزمایش Epsilon Sensitivity</h1>
  <h3 style="font-size: 18px; color: #374151;">روند کار</h3>
  <p>
    با ثابت نگه داشتن تعداد نقاط، ابعاد و <code>MinPts</code>، مقدار <code>epsilon</code> تغییر داده شد
    تا اثر شعاع همسایگی روی تعداد clusterها، نویز و زمان اجرا مشخص شود.
  </p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 1440
Dimensions = 3
MinPts = 5
ε = 0.10 تا 1.50</pre>

  <h3 style="font-size: 18px; color: #374151;">نتیجه نهایی</h3>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>در <code>ε = 0.10</code> سه نقطه noise شدند.</li>
    <li>از <code>ε ≥ 0.20</code> در dataset آزمایش noise اصلی از بین رفت.</li>
    <li>زمان اجرا فقط مقدار محدودی تغییر کرد.</li>
    <li>در brute-force، تغییر epsilon تعداد scanهای کلی را خیلی تغییر نمی‌دهد.</li>
  </ul>
  <p><b>نتیجه:</b> epsilon بیشتر روی <b>رفتار clustering</b> اثر دارد تا روی حافظه و زمان نسخه brute-force.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۳. آزمایش MinPts Sensitivity</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 1440
ε = 0.20
MinPts = 2, 5, 10, 20, 50, 100, 200</pre>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li><code>MinPts = 2</code>: بیش از حد permissive بود.</li>
    <li><code>MinPts = 5</code>: بهترین رفتار را در dataset آزمایش ایجاد کرد.</li>
    <li><code>MinPts = 10</code>: باعث fragmentation شد.</li>
    <li><code>MinPts = 20</code>: بیشتر نقاط noise شدند.</li>
    <li><code>MinPts ≥ 50</code>: تمام نقاط noise شدند.</li>
  </ul>
  <p><b>نتیجه:</b> <code>MinPts</code> مستقیماً سخت‌گیری density را کنترل می‌کند.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۴. آزمایش Density Sensitivity</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Dense   → radius = 0.25
Medium  → radius = 0.60
Sparse  → radius = 1.20</pre>
  <ul style="padding-right: 22px; font-size: 16.5px;">
    <li>Dense بیشترین cluster expansion و بیشترین زمان را داشت.</li>
    <li>Medium رفتار میانی داشت.</li>
    <li>Sparse به دلیل کم بودن نقاط core، بسیار سریع‌تر شد.</li>
  </ul>
  <p><b>نتیجه:</b> تراکم داده روی <b>زمان واقعی اجرا</b> اثر زیادی دارد.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۵. آزمایش Spatial Grid Optimization</h1>
  <p>بهترین speedup:</p>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">N = 720
ε = 0.10
MinPts = 5

Baseline = 111.411 ms
Optimized total = 5.186 ms
Speedup = 21.48×</pre>
  <p>در <code>N = 1440</code>:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 837.438 ms
Optimized total = 56.784 ms
Speedup = 14.75×</pre>
  <p><b>نتیجه:</b> Spatial Grid مهم‌ترین بهینه‌سازی کل پروژه بود.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۶. آزمایش Dual-Core — روش A</h1>
  <p>تشخیص <code>core-point</code> بین Core 0 و Core 1 تقسیم شد، ولی cluster expansion روی یک هسته ماند.</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 693.067 ms
Dual-Core = 685.656 ms
Speedup = 1.01×
Improvement = 1.07%</pre>

  <h1 style="font-size: 24px; color: #111827;">۷. آزمایش Dual-Core — روش B</h1>
  <p>در هر neighborhood query، Core 0 نصف نقاط و Core 1 نیمه دیگر را بررسی کردند.</p>
  <pre style="direction:ltr; text-align:left; background:#7f1d1d; color:#fef2f2; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline = 645.262 ms
Dual-Core = 787.090 ms
Speedup = 0.82×  ⚠️ کندتر شد</pre>
  <p><b>نتیجه:</b> هزینه synchronization بیشتر از سود parallelization شد.</p>

  <h1 style="font-size: 24px; color: #111827;">۸. آزمایش Dual-Core — روش C</h1>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Single-Core Grid = 37.085 ms
Dual-Core Grid   = 36.703 ms
Speedup = 1.01×</pre>

  <h1 style="font-size: 24px; color: #111827;">۹. آزمایش Dual-Core — روش D ⭐</h1>
  <p>در این روش خود <b>cluster expansion</b> موازی شد.</p>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Expansion speedup = 1.05×
End-to-end speedup = 1.03×
Improvement = 2.88%
Different labels = 0</pre>
  <p><b>نتیجه:</b> روش D بهترین روش Dual-Core آزمایش‌شده بود.</p>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۰. آزمایش Three-Sensor Simulation</h1>
  <p>یک pipeline شبیه کاربرد واقعی ساخته شد:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Sensor 1 ─┐
Sensor 2 ─┼→ Ingest/Parsing → Fusion → Spatial Grid
Sensor 3 ─┘                         ↓
                                   DBSCAN
                                     ↓
                              Human Analysis</pre>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">2 simulated humans
        ↓
2 DBSCAN clusters
        ↓
2 human-like clusters  ✅</pre>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۱. آزمایش Real-Time Limit — 07B</h1>
  <p>بودجه هر frame در نرخ <b>10 FPS</b>:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">100 ms</pre>

  <h3 style="font-size: 18px; color: #374151;">نتیجه موفق ✅</h3>
  <pre style="direction:ltr; text-align:left; background:#065f46; color:#ecfdf5; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">104 points / human / sensor
660 points / frame
6600 points / second @ 10 FPS

Average pipeline = 89.357 ms
Maximum pipeline = 96.496 ms

Detection stable = YES (2 humans)</pre>

  <h3 style="font-size: 18px; color: #374151;">اولین configuration ناموفق ❌</h3>
  <pre style="direction:ltr; text-align:left; background:#7f1d1d; color:#fef2f2; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">108 points / human / sensor
684 points / frame
Maximum pipeline = 101.247 ms
Frame overruns = 5 / 12</pre>

  <p>مرز عملی آزمایش‌شده:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">660 points/frame   ✅
684 points/frame   ❌</pre>

  <p>زمان متوسط بخش‌ها در configuration موفق:</p>
  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Ingest + fusion ≈ 3.396 ms
Grid + DBSCAN   ≈ 84.708 ms
Human analysis  ≈ 1.250 ms</pre>

  <hr style="border: none; border-top: 1px solid #e5e7eb; margin: 28px 0;">

  <h1 style="font-size: 24px; color: #111827;">۱۲. جمع‌بندی نهایی پروژه</h1>

  <pre style="direction:ltr; text-align:left; background:#111827; color:#f9fafb; padding:14px 18px; border-radius:8px; font-family:'Fira Code',monospace; font-size:14.5px;">Baseline DBSCAN
      ↓
Parameter Analysis
      ↓
Spatial Grid
      ↓
Dual-Core Experiments
      ↓
3-Sensor Simulation
      ↓
Human Movement + Fusion
      ↓
Real-Time Limit Test</pre>

  <h3 style="font-size: 20px; color: #1d4ed8;">نتیجه اصلی پروژه</h3>
  <blockquote style="background: #eff6ff; border-right: 5px solid #3b82f6; padding: 18px 22px; border-radius: 8px; font-size: 17px; color: #1e3a8a; margin: 16px 0;">
    روی ESP32-S3، بهینه‌سازی الگوریتمی جستجوی همسایه بسیار مهم‌تر از parallelization ساده است.
    Spatial Grid بزرگ‌ترین بهبود را ایجاد می‌کند (<b>21.48×</b>) و Dual-Core زمانی مفید است که
    روی بخش محاسباتی مناسب، مانند cluster expansion، اعمال شود. در pipeline چندسنسوره شبیه‌سازی‌شده،
    سیستم فعلی در نرخ 10 FPS حداقل <b>660 نقطه در هر frame</b> را به‌صورت real-time و با تشخیص پایدار
    دو انسان پردازش می‌کند.
  </blockquote>

  <h3 style="font-size: 20px; color: #b45309;">محدودیت فعلی</h3>
  <p>
    آزمایش Real-Time هنوز با داده‌های سنسوری شبیه‌سازی‌شده انجام شده است و دریافت فیزیکی سه آنتن واقعی،
    baud rate، packet format، buffering و overhead واقعی UART هنوز وارد benchmark نشده‌اند.
    بنابراین عدد 6600 point/s یک <b>نتیجه تجربی برای pipeline شبیه‌سازی‌شده فعلی</b> است،
    نه سقف نظری نهایی ESP32-S3.
  </p>

</div>
"""

# ============ 3) نمایش در Notebook ============
display(HTML(html_content))

# ============ 4) ذخیره به فایل HTML مستقل ============
full_html = f"""<!DOCTYPE html>
<html lang="fa" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>DBSCAN Benchmark Report - ESP32-S3</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Vazirmatn:wght@400;600;800&display=swap" rel="stylesheet">
<style>
  body {{
    background: #eef2f7;
    margin: 0;
    padding: 20px;
    font-family: 'Vazirmatn', 'Segoe UI', Tahoma, sans-serif;
  }}
</style>
</head>
<body>
{html_content}
</body>
</html>"""

output_path = Path("dbscan_report.html")
output_path.write_text(full_html, encoding="utf-8")

print(f"✅ گزارش با موفقیت ذخیره شد")
print(f"📁 مسیر فایل: {output_path.resolve()}")
print(f"📊 حجم فایل: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("💡 برای PDF: فایل HTML را در Chrome باز کن و Ctrl+P بزن")

✅ گزارش با موفقیت ذخیره شد
📁 مسیر فایل: C:\Users\iCFixer.ir\Documents\PlatformIO\Projects\esp32-s3-final-project-phase-1\results\dbscan_report.html
📊 حجم فایل: 14.7 KB

💡 برای PDF: فایل HTML را در Chrome باز کن و Ctrl+P بزن
